<a href="https://colab.research.google.com/github/jonay-lab/Assignment_Practice/blob/main/Assignment_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment_13:Generative AI Essentials**

**objective:**

understanding of Generative AI basics, focusing on its key concepts, the architecture of models like Generative Pre-trained Transformers (GPTs), and their practical applications.

**Import necessary libraries**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install transformers torch

In [3]:
!pip install -q transformers datasets accelerate torch
import torch
from datasets import load_dataset
from transformers import (GPT2Tokenizer,GPT2LMHeadModel,DataCollatorForLanguageModeling,Trainer,TrainingArguments,)
import requests

**Loading the data set**

The dataset is a public-domain text from Project Gutenberg. It contains the complete text of a book, which is used to fine-tune the GPT-2 model. During training, the model learns the vocabulary, sentence structure, writing style, and language patterns from the book so it can generate similar text

In [4]:
url = "https://www.gutenberg.org/files/7256/7256-0.txt"
response = requests.get(url)
response.raise_for_status()

with open("gift_of_the_magi.txt", "wb") as f:
    f.write(response.content)

dataset = load_dataset("text", data_files={"train": "gift_of_the_magi.txt"})

print(dataset["train"][:2])

Generating train split: 0 examples [00:00, ? examples/s]

{'text': ['\ufeffThe Project Gutenberg eBook of The Gift of the Magi, by O. Henry', '']}


In [5]:
print(dataset["train"][0]["text"][:500])

﻿The Project Gutenberg eBook of The Gift of the Magi, by O. Henry


**Loading GPT2 Tokenizer**

The GPT-2 tokenizer converts text into tokens (numerical representations) that the model can process. The pre-trained GPT-2 model has already learned general language patterns from a large corpus of text. During fine-tuning, it adapts this knowledge to the selected Project Gutenberg book, improving its ability to predict the next token and generate coherent, grammatically correct text in a similar style.

In [6]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Tokenize the data set**

this step converts the book text into tokens that gpt-2 can understand. tokenization transforms words and punctuation into numerical IDS, making the text suitable as input for the model during funi-tuning. this allows gpt-2 to learn patterns and relationships between tokens in the dataset.

In [7]:
def tokenize_function(examples):
    return tokenizer(examples["text"],truncation=True,max_length=128)
tokenized_dataset = dataset.map(
    tokenize_function,batched=True,remove_columns=["text"])

tokenized_dataset = tokenized_dataset.filter(lambda x: len(x["input_ids"]) > 0)

Map:   0%|          | 0/606 [00:00<?, ? examples/s]

Filter:   0%|          | 0/606 [00:00<?, ? examples/s]

**Creating Data collator**

the data collator prepares batches of tokenized text for training. it organizes the input data into the correct format and creates the labels required for GPT-2 , allowing the model to learn by predicting the next token in each sequence.

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

**Training Arguments**


the training arguments define how gpt-2 is fine-tuned on the project the dataset.

In [10]:
training_args = TrainingArguments(
    output_dir="./gpt2-magi",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=100
)


**Fine-tune GPT-2**

In this step , the pre trained gpt-2 model is trained on the data set.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.398627
200,3.138974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=246, training_loss=3.2396368631502477, metrics={'train_runtime': 371.2779, 'train_samples_per_second': 1.325, 'train_steps_per_second': 0.663, 'total_flos': 4091874048000.0, 'train_loss': 3.2396368631502477, 'epoch': 1.0})

**Text Generation Function**

this function will accepts a user-prompt and converts it into tokens before passing it to the fine turned GPT-2 model.

In [25]:
def generate_text(prompt, max_length=100, temperature=1.0, top_k=50, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [26]:
print(generate_text("Her eyes were shining brilliantly"))

Her eyes were shining brilliantly. It was not a dream she had, as she was so sure it was to look. The girl looked at her with the eyes that were full of joy and had always been, in the way that this beautiful girl was always looked, and she was just glad that now she had done it with a little and was about to. She had been a beautiful, nice, little girl, and it was a happy one of them, a joy, to be at this kind


** Controlling output length**

using max_length we can control number of token generate by GPT-2

In [27]:
print(generate_text("Della looked at hair",max_length=50))


Della looked at hairless young and pale, with bright lashes and a strange look in the way of a little girl. It had been a long time since she had seen any hair of that kind on her now and it was as bright as any


In [28]:
print(generate_text("Della looked at her hair",max_length=150))

Della looked at her hair and her face looked from the back of the room to the window. She looked at the clock on the desk. "Tis time had been set at 1:00 a.m. the last of the day, that was to say, she and then she found it was like midnight before that. A year ago. She had been out at the corner of the room, with her hands out her door, watching a woman walk down by the window. The clock was on the other side of the window in the corner. It was her face, and then she had to see where she had been, and she looked at it, and she would do it. And she said, 'Goodness!' And she


**Adjusting Temperature**

controls randoms of gpt-2's text generations

In [29]:
print(generate_text("Jim stopped",temperature=0.3))


Jim stopped and looked at the door. It was a door that had been opened by the door of the house. It was a door that had been opened by the housekeeper. It was a door that had been opened by the housekeeper. The door was a door that had been opened by the housekeeper. It was a door that had been opened by the housekeeper. It was a door that had been opened door that had been opened by the housekeeper. It was a door that had


In [18]:
print(generate_text("Jim stopped",temperature=1.2))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Jim stopped the car and waved to her, to her room on the other side of the room—the door to her bathroom on the side of the floor of the office. "Oh that looks beautiful on the inside!" whispered the girl as the door shut in the window. That would make it. Now, that's not all—she turned her head back to gaze with her full-blown eyes in the darkness. She looked up at his face without her permission, but he was up,


**Using Top_k and Top_P**

using top_k and top_p parametes control how gpt-2 selects next token during text generation.

In [30]:
print(generate_text("jim returned home",top_k=30,top_p=0.85))

jim returned home, and he was now in his early twenties. He was walking down the street, his eyes closed and he did not notice any difference. He was just as much a man as the other two. The last thing he remembered was how he had been walking. It was a beautiful and he had walked a fine, to be walked a fine, beautiful, long, smooth and graceful walk, he had not said. And now it was the dark. He had seen the great.


In [31]:
print(generate_text("jim returned home",top_k=80,top_p=0.95))

jim returned home to the house at which it was located. He was the mother of any grandson which is now referred to—not the eldest to the last and not the youngest—the mother who was to give birth to her grandson and grandchildren after his father's birth. She was a sister of such a daughter as had never been. "And what a beautiful, beautiful, beautiful girl is she to me," he said to his children, and to them he seemed to have no doubt, he


**Prompt Engineering **

In [32]:
print(generate_text("write how people look at each other"))

print(generate_text( "what is the best day walk"))

print(generate_text( "Explain artificial intelligence in simple words."))

write how people look at each other, and what they look for, and how you look at them. It was an impossible task that I had just learned in school. It was a new school. It was not a new world. I knew it had a new way of being a person in the world—and now I needed it. And the new world, and I needed it was just the best way to do it. It had been the new world of me. And now, all through
what is the best day walk to work you can do when you are first started out in the hobby! If you don't know how to get started in creating your own work, there are a few things you can do with a full-time job as a hobbyist on their site. The easiest way to learn to do it is to become an amateur at first with a website and then learn to become a hobbyist. As a hobbyist, you can find lots of great sites, the ones
Explain artificial intelligence in simple words. It may be a work in progress, that in no way resembles the work of other people, and that its meaning is not altered or confirmed by any external 

In [35]:
print(" Interactive prompt engineering")
user_prompt = input("Enter your own prompt: ")
print("\nGenerated text:")
print(generate_text(user_prompt, max_length=200))

 Interactive prompt engineering
Enter your own prompt: it is an end

Generated text:
it is an end in the world of unrequited generosity toward the free expression of thoughts, no matter how great or small. It is a work of the individual can do with any work, any purpose, without charge who can buy it. It is a sign of the world's hard work that no one can say what a good book must be like. What is not to like?"


She seemed to be about to do with this and, in the end, would no sooner finish her sentence than a hundred, and she went on as usual. She had the longest, most rueful smile on her face. She looked like her parents she had ever seen, like an overblown dainty. And at last she was out for some time. There was a pause and then, and now a flash and a wistful little laugh. She looked with unaided curiosity a little bit on the face, and then a little more, and then she remembered again the strange
